In [13]:
import numpy as np
from scipy.stats import norm

### **iid data**

**Gaussian distribution**

$$ 
X_1, \ldots, X_n \overset{\text{iid}}{\sim} \mathcal{N}(\mu, \sigma^2)
$$

In [14]:
def generate_normal (n, mean=0.0, sd=1.0, seed=None):
   
    rng = np.random.default_rng(seed)
    x = rng.normal(loc=mean, scale=sd, size=n)
    
    return x

**Gaussian mixture distribution**

$$
X_1,\ldots,X_n \overset{\text{iid}}{\sim}
w\,\mathcal{N}(\mu_1,\sigma_1^2)
+
(1-w)\,\mathcal{N}(\mu_2,\sigma_2^2),
\qquad 0 \leq w \leq 1
$$

In [15]:
def generate_gaussian_mixture(n, weights=(0.5, 0.5), means=(-2.0, 2.0), sds=(0.5, 1.0), seed=None ):
    
    rng = np.random.default_rng(seed)
    components = rng.choice(len(weights), size=n, p=weights)     # 두 개(len(weight)))의 선택지 중 뽑을 확률 각각 weight 로 n개를 뽑음

    x = rng.normal(loc=means[components], scale=sds[components])

    return x

### **Dependent time series data**

**AR(1)**

$$
X_t = \phi_1 X_{t-1} + \varepsilon_t,
$$

$$
\varepsilon_t \overset{\text{iid}}{\sim} \mathcal{N}(0,\sigma^2),
$$

$$
\qquad t = 1,\ldots,n.
$$

If $|\phi_1| < 1$, a stationary distribution exists.

In [16]:
def generate_ar1(n, phi=0.7, sigma=1.0, x0=0.0, seed=None):

    rng = np.random.default_rng(seed)
    x = np.zeros(n)
    epsilon = rng.normal(loc=0.0, scale=sigma, size=n)
    
    for t in range(1, n):
        x[0] = x0
        x[t] = phi * x[t-1] + epsilon[t]

    return x

**GARCH(1,1)**

$$
X_t = \sigma_t \varepsilon_t,
$$

$$
\sigma_t^2
=
\omega
+
\alpha X_{t-1}^2
+
\beta \sigma_{t-1}^2,
$$

$$
\varepsilon_t \overset{\text{iid}}{\sim} \mathcal{N}(0,1),
$$

$$
t=1,\ldots,n.
$$

$X_t$ : return,

$\sigma_t^2$ : conditional variance,

$\epsilon_t$ : standard normal innovation.

If $\alpha + \beta < 1$, a stationary variance exists.

In [17]:
def generate_garch11(n, omega=0.1, alpha=0.1, beta=0.8, seed=None):
  
    rng = np.random.default_rng(seed)

    x = np.zeros(n)
    sigma2 = np.zeros(n)
    epsilon = rng.normal(loc=0.0, scale=1.0, size=n)

    # unconditional variance
    sigma2[0] = omega / (1 - alpha - beta)
    x[0] = np.sqrt(sigma2[0]) * epsilon[0]

    for t in range(1, n):
        sigma2[t] = omega + alpha * x[t - 1] ** 2 + beta * sigma2[t - 1]
        x[t] = np.sqrt(sigma2[t]) * epsilon[t]

    return x, sigma2